# Handwritten Digit Recognition (MNIST)

A feedforward/convolutional neural network (Keras/TensorFlow) that classifies
handwritten digits (0-9).

MNIST stands for Modified National Institute of Standards and Technology.

Run the cells top to bottom. Use the **Python (handwritten-nn)** kernel so the
project's dependencies (tensorflow, pandas, numpy, scikit-learn, matplotlib) are
available.

## Step 1: Import libraries

Before starting, we need to import the necessary libraries for data manipulation, visualization, and model building.

In [ ]:
print("Importing modules...")
import os
# Silence TensorFlow's informational/warning logs like below:
#   WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
#   I0000 ... oneDNN custom operations are on. ... set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
# These MUST be set before tensorflow is imported, since the C++ backend reads them at import time.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"   # 0=all, 1=no INFO, 2=no WARNING, 3=no ERROR
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"  # removes the oneDNN custom-operations notice
# --- Core data / numeric libraries ---
import re
import numpy as np                 # numerical arrays and math (reshaping, argmax, normalization)
import pandas as pd                # reading the CSV dataset and holding it as a DataFrame
import matplotlib.pyplot as plt    # plotting training/validation accuracy and showing sample digits
# --- Data splitting ---
from sklearn.model_selection import train_test_split  # split data into train/validation sets
# --- Keras model container ---
from tensorflow.keras.models import Sequential  # linear stack of layers, one after another
# --- Dense (fully-connected) building blocks ---
from tensorflow.keras.layers import Dense, Flatten, Input
#   Input   - declares the input shape, e.g. (28, 28, 1) for one grayscale image
#   Flatten - turns a 2D/3D image into a 1D vector (28x28x1 -> 784) for Dense layers
#   Dense   - fully-connected layer; every neuron connects to all inputs (hidden + output layers)
# --- Convolutional building blocks (for a CNN, better at images) ---
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout
#   Conv2D       - convolutional layer; learns local spatial features (edges, curves) from the 2D image
#   MaxPooling2D - downsamples feature maps (keeps the strongest signal, shrinks size, adds robustness)
#   Dropout      - regularization; randomly zeroes some neurons during training to reduce overfitting
# --- Label preprocessing ---
from tensorflow.keras.utils import to_categorical  # one-hot encode labels: 3 -> [0,0,0,1,0,0,0,0,0,0]
print("Import done.")
# Global methods
def shift_pixel_name(col):
    # match "pixel" followed by a number; bump the number by 1
    m = re.fullmatch(r"pixel(\d+)", col)
    if m:
        return f"pixel{int(m.group(1)) + 1}"
    return col  # leave non-pixel columns (e.g. "label") unchanged

## Step 2: Load and explore the dataset

We will load the dataset and inspect its structure to understand the features and labels.

It's a classic dataset of handwritten digits (0-9) widely used for training and testing image classification models.

Each image is 28x28 pixels (784 pixels), showing a single handwritten digit. It's often called the "Hello World" of machine learning because it's small, well-understood, and quick to train on, making it ideal for learning.

You can download the dataset from https://media.geeksforgeeks.org/wp-content/uploads/20250407132659565012/Train.csv

Reads the CSV file into a DataFrame, with a note that column 0 is the label (the digits) and columns 1-784 are pixels.

In [ ]:
pd.set_option("display.max_columns", None)   # None = no limit
print("Loading training data from Train.csv file...")
train_data = pd.read_csv('datasets/Train.csv')
# .shape is (rows, columns) -> e.g. (42000, 785): 42000 images x (1 label + 784 pixels).
print("Done. Shape of train_data:", train_data.shape)
train_data = train_data.rename(columns=shift_pixel_name)
train_data.info()
train_data.head(5)

In [ ]:
# Split each row into features (X) and label (y). iloc is integer-position indexing:
#   [:, 1:] -> all rows, columns 1..end  -> the 784 pixel values (the inputs)
X = train_data.iloc[:, 1:]
# X now has 784 columns (the label column has been dropped) -> (42000, 784)
print("Shape of X after separating features:", X.shape)
X.head(5)

## Step 3: Preprocess the data

Raw data often needs cleaning and formatting before it can be fed into a neural network. Let's preprocess the data to make it ready for training.

- Convert all pixel values to numeric format and replace any missing values with 0.
- Next we [normalize](https://www.geeksforgeeks.org/machine-learning/how-to-normalize-data-using-scikit-learn-in-python/) the pixel values to the range [0, 1] by dividing them by 255.0. This helps the model learn faster.
- Finally we reshape the data to include a channel dimension making it compatible with neural networks.

In [ ]:
# Force every cell to a number.
# errors='coerce' means any value that can't be converted (blank cell, stray text like "?") becomes NaN instead of raising an error.
# This guarantees the data is numeric before the model sees it;
# the NaNs produced here are cleaned up by the fillna(0) line below.
X = X.apply(pd.to_numeric, errors='coerce')
#  X.fillna(0) scans the whole DataFrame and replaces every NaN with 0:
# before:  [12, NaN, 255, NaN, 3, ...]
# after:   [12,   0, 255,   0, 3, ...]
# single total count of NaN across the whole DataFrame
print ("Number of NaN is X:", X.isna().sum().sum()) # first .sum() → NaN count per column, second .sum() → grand total
X = X.fillna(0)  # This line replaces any missing values in your pixel data with 0.
display(X.head(5))      # nicer rendering in Jupyter

In [ ]:
# Remark: cell can be executed once!
# / 255.0 - divides every pixel by 255.0. Because of NumPy broadcasting, this single division applies to all values at once, no loop needed.
# Since pixel values run 0-255, dividing by 255 rescales them to the 0.0-1.0 range:
# 0 (black)   -> 0.0
# 128 (gray)  -> 0.502
# 255 (white) -> 1.0
# Neural networks train much better when inputs are small and in a consistent range.
X = X.values / 255.0
# Below line is for nicer rendering in Jupyter
display(pd.DataFrame(X).head(5)) # wrap first, THEN .head()

In [ ]:
# Remark: cell can be executed once!
# reshape(-1, 28, 28, 1) turns the (42000, 784) array into (42000, 28, 28, 1):
#  -1  number of images                42000
#  28  image height (rows of pixels)    28
#  28  image width (columns of pixels)  28
#   1  channels (color depth)           1 = grayscale
# Keras image layers expect input shaped (batch, height, width, channels).
X = X.reshape(-1, 28, 28, 1)
print("Shape of X after reshaping:", X.shape)

In [ ]:
# Show first row
# X[0]

In [ ]:
#   [:, 0]  -> all rows, column 0 -> the digit label 0-9 (the target)
y_orig = train_data.iloc[:, 0]
y_orig.info()

In [ ]:
counts = y_orig.value_counts().sort_index()
plt.figure(figsize=(8, 5))
bars = plt.bar(counts.index, counts.values, color="steelblue")
plt.xlabel("Digit")
plt.ylabel("Number of samples")
plt.title("Distribution of labels (0-9)")
plt.xticks(range(10))
# write the exact count on top of each bar
for bar, c in zip(bars, counts.values):
    plt.text(bar.get_x() + bar.get_width() / 2, c, str(c),
             ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

## Step 4: One-Hot encode the labels

`y` is a 1D array of 42000 integers, each between 0 and 9. Neural networks work best when labels are in a format called [one-hot encoding](https://www.geeksforgeeks.org/machine-learning/ml-one-hot-encoding/). Let's convert our labels into this format.

`to_categorical(y, num_classes=10)` turns each integer into a length-10 vector that's all zeros except for a single 1 at the position matching the digit:

| Original label | One-hot vector |
|----------------|----------------|
| 0 | [1, 0, 0, 0, 0, 0, 0, 0, 0, 0] |
| 1 | [0, 1, 0, 0, 0, 0, 0, 0, 0, 0] |
| 2 | [0, 0, 1, 0, 0, 0, 0, 0, 0, 0] |
| 3 | [0, 0, 0, 1, 0, 0, 0, 0, 0, 0] |
| 4 | [0, 0, 0, 0, 1, 0, 0, 0, 0, 0] |
| 5 | [0, 0, 0, 0, 0, 1, 0, 0, 0, 0] |
| 6 | [0, 0, 0, 0, 0, 0, 1, 0, 0, 0] |
| 7 | [0, 0, 0, 0, 0, 0, 0, 1, 0, 0] |
| 8 | [0, 0, 0, 0, 0, 0, 0, 0, 1, 0] |
| 9 | [0, 0, 0, 0, 0, 0, 0, 0, 0, 1] |

After this line, `y` changes shape from (42000,) to (42000, 10). `num_classes=10` fixes the vector length at 10 (digits 0-9).

In [ ]:
print("First 5 labels:")
display(pd.DataFrame(y_orig).head(5)) # wrap first, THEN .head()
y = to_categorical(y_orig, num_classes=10)
print("Shape of y after one-hot encoding:", y.shape)
print("First 5 one-hot encoding lables:")
display(pd.DataFrame(y).head(5)) # wrap first, THEN .head()

## Step 5: Split the data

To evaluate our model effectively we split the data into a training set and a validation set. Here we use 80% for training and 20% for validation.

- `test_size=0.2` - hold out 20% for validation. With 42000 images that's a 33600/8400 split.
- `random_state=42` - seeds the shuffle so you get the exact same split every run. The number 42 is arbitrary; any fixed integer works. Without it, each run produces a different split, making results non-reproducible.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
# X_train images to train on:   33600 (80%)
# X_val   images to validate on: 8400 (20%)
# y_train labels for the training images:  33600 (80%)
# y_val   labels for the validation images: 8400 (20%)
print("X_train shape:", X_train.shape)
print("X_val   shape:", X_val.shape)

## Step 6: Build the neural network model

Now let's define the architecture. The original dense model (`Flatten -> Dense(128) -> Dense(64) -> Dense(10)`) flattens the image and throws away all 2D spatial structure.

Switching to a **CNN** keeps the 28x28 grid and learns local patterns (edges, curves, loops) that are far more digit-relevant, which moves MNIST from ~97-98% into the 99%+ range. This notebook uses the CNN version:

- `Conv2D` layers learn spatial features, `MaxPooling2D` downsamples, `Dropout` regularizes.
- `Flatten` then feeds a `Dense(128)` hidden layer and a `Dense(10, softmax)` output.

We compile with the Adam optimizer, categorical cross-entropy loss, and accuracy as the reported metric.

In [ ]:
# Original dense-only model (kept for reference):
# model = Sequential([
#     Input(shape=(28, 28, 1)),
#     Flatten(),
#     Dense(128, activation='relu'),
#     Dense(64, activation='relu'),
#     Dense(10, activation='softmax')
# ])
model = Sequential([
    Input(shape=(28, 28, 1)),
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(10, activation='softmax'),
])
# Configure the model for training. compile() wires up three things:
#   optimizer='adam'                 -> algorithm that updates weights to reduce error (adaptive default)
#   loss='categorical_crossentropy'  -> quantity minimized; distance from one-hot true labels
#   metrics=['accuracy']             -> reported (not optimized): % of correct predictions
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
# Print a table of the layers, output shapes, and parameter counts (informational only).
model.summary()

## Step 7: Train the model

With the model defined it's time to train it on the training data. The model trains for 10 epochs (full passes over the training set), processing the data in batches of 32 for efficiency.

- `epochs=10` - the model sees the entire training set 10 times; too few underfits, too many overfits.
- `batch_size=32` - weights update after each batch of 32 images (33600/32 = 1050 updates per epoch).
- `validation_data=(X_val, y_val)` - after each epoch the model is measured on the 8400 held-out images (never used to update weights), giving an honest read on generalization.

**Note:** this cell is the slow one - it does the actual training.

In [ ]:
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_val, y_val))
# --- Comparison: read the last epoch's validation metrics straight from history ---
# history.history stores one value per epoch. The LAST entry ([-1]) is the final
# epoch's validation loss/accuracy -- the same model/data that model.evaluate() re-measures.
hist_val_loss = history.history['val_loss'][-1]
hist_val_accuracy = history.history['val_accuracy'][-1]
print("--- from history (last epoch, no re-computation) ---")
print(f"history val_loss:     {hist_val_loss:.4f}")
print(f"history val_accuracy: {hist_val_accuracy * 100:.2f}%")

## Step 8: Evaluate the model

Once training is complete we evaluate performance on the validation set and plot the training/validation accuracy to see how well the model learned over time (helps spot overfitting).

- **Accuracy** - `correct predictions / total predictions`. Accuracy: 0.9724 means 97.24% of validation images classified correctly.
- **Loss** - how far the predicted probabilities are from the truth (categorical cross-entropy). Lower is better; a confidently wrong prediction has very high loss.

The plot below shows one X-axis tick per epoch.

In [ ]:
print("--- from model.evaluate (re-runs inference over X_val) ---")
val_loss, val_accuracy = model.evaluate(X_val, y_val)
print(f"Validation Loss:     {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy * 100:.2f}%")
# In a notebook the plot renders inline (no separate window).
plt.figure('Model Accuracy per Epoch')
# num_epochs is taken from the history length so it always matches the number of
# epochs actually trained. epochs = 1, 2, ... num_epochs (one per recorded point).
num_epochs = len(history.history['accuracy'])
epochs = range(1, num_epochs + 1)
# Plot against the explicit 1-based epoch numbers so the X-axis starts at 1 (not 0).
plt.plot(epochs, history.history['accuracy'], label='Training Accuracy')
plt.plot(epochs, history.history['val_accuracy'], label='Validation Accuracy')
# One X-axis tick per epoch: 1, 2, ... num_epochs.
plt.xticks(epochs)
plt.title('Model Accuracy per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

## Step 9: Save the trained model (architecture + weights)

The stored model will be used later for prediction.

Persist the fully trained model to a single `.keras` file. This bundles everything needed to reuse it later: the architecture (layers), the learned weights, and the compile settings (optimizer/loss/metrics). Other scripts (`predict.py`, the UIs) then just `load_model('mnist_model.keras')` - no need to retrain. The `.keras` extension is Keras 3's native zip-based save format.

In [ ]:
model.save('mnist_model.keras')
print("Saved trained model to mnist_model.keras")

## Step 10: Make predictions

Let's use the trained model to make predictions on new data. We load the test data, preprocess it the same way as the training data, and feed it into the model to get predictions.
You can download the test dataset from: https://media.geeksforgeeks.org/wp-content/uploads/20250707130706513705/test.zip

The loop below spot-checks the first 5 predictions by drawing each image with its predicted label.

In [ ]:
# Load the unlabeled test set (pixels only, no label column) into a DataFrame.
print("Loading training data from test.csv file...")
test_data = pd.read_csv('datasets/test.csv')
# .shape is (rows, columns) -> e.g. (5600, 784): 5600 images x 784 pixels
print("Done. Shape of test_data:", test_data.shape)
test_data = test_data.rename(columns=shift_pixel_name)
test_data.info()
test_data.head(5)

In [ ]:
# Same preprocessing as training: normalize pixels from 0-255 to 0.0-1.0.
# .values converts the DataFrame to a NumPy array first.
X_test = test_data.values / 255.0
# Reshape flat 784-pixel rows into (num_images, 28, 28, 1) to match the model input.
X_test = X_test.reshape(-1, 28, 28, 1)
# Run the trained model on all test images. Each output row is 10 softmax probabilities.
predictions = model.predict(X_test)
# Collapse each 10-probability row to a single predicted digit (index of the highest value).
predicted_labels = np.argmax(predictions, axis=1)
# Visually spot-check the first 5 predictions: show each image with its predicted label.
for i in range(5):
    plt.figure()
    plt.imshow(X_test[i].reshape(28, 28), cmap='gray')  # (28,28,1) -> 2D grayscale
    plt.title(f"Predicted: {predicted_labels[i]}")
    plt.axis('off')
    plt.show()